# Genomic Intelligence

![Genomic Intelligence](https://proto-bio.github.io/proto-assets/images/tool/genomic_intelligence/hero.png)

> This notebook was developed by [Genomic Intelligence](https://genomicintelligence.ai/)

Example use cases of the seven tools over the hosted Genomic Intelligence `/v1` API: promoter, splice,
enhancer, chromatin and annotation scoring, gene-expression prediction, and a
workflow that finds genes in a locus and scores expression for each one.

Inference runs on the vendor's service, so there is no model to download and no
GPU required. Set `GI_API_KEY` before running; request a key at
<https://genomicintelligence.ai>.

Research and development use. Not for clinical or diagnostic decisions.

**Outputs below were produced against Genomic Intelligence service version
`2026.08.19.5`.** Predictions move as models are retrained; re-run the notebook
rather than treating the printed numbers as current.

In [1]:
from proto_tools.utils.notebook_docs import (
    display_api_reference,
    display_available_tools,
    display_doc_link,
    display_docs_section,
    display_overview,
)

display_doc_link("genomic_intelligence")
display_overview("genomic_intelligence")
display_docs_section("genomic_intelligence", "Background")

# Genomic Intelligence

[Genomic Intelligence](https://genomicintelligence.ai) serves transformer DNA language models that score regulatory function directly from sequence. This toolkit exposes seven tools over its [`/v1` REST API](https://docs.genomicintelligence.ai): `gi-promoter` (promoter regions), `gi-splice` (donor and acceptor sites), `gi-enhancer` (enhancer activity), `gi-chromatin` (chromatin state), `gi-annotation` (de-novo transcripts), `gi-expression` (expression from a TSS window), and `gi-find-genes-and-predict-expression` (both, in one call). Inference runs on the vendor's service, so no weights are downloaded and no GPU is needed.

Sequence-to-function models predict regulatory readouts from DNA alone, without an assay. Genomic Intelligence hosts a family of them behind one API: promoter and enhancer classifiers, a splice-site model, a chromatin-state panel spanning accessibility, transcription-factor occupancy and histone marks, a structure-aware gene finder, and an expression model conditioned on experimental context. Each task is a separate published operation with its own request schema and its own minimum input length, so bounds are per task rather than global.

Every tool here is a thin HTTPS client. A request carries the sequence and the task's options; the service resolves which model version to run, so `model` is left unset by default and the alternatives are enumerable through `GET /v1/tasks/{task}/models`. Delivery is a per-request choice on every endpoint: omitting the `Prefer` header returns the result synchronously, while `respond_async` returns a job id to poll. Coordinates in tool outputs are 0-based with exclusive ends, following the genomics interval convention used elsewhere in `sequence_scoring` rather than the 1-based residue numbering used across the rest of proto-tools.

## Available tools

In [2]:
display_available_tools("genomic_intelligence")

- **`run_gi_annotation()`** — Find genes and transcripts de novo in raw DNA via the hosted Genomic Intelligence API
- **`run_gi_chromatin()`** — Predict chromatin accessibility, TF occupancy and histone marks via the hosted Genomic Intelligence API
- **`run_gi_enhancer()`** — Predict developmental and housekeeping enhancer activity via the hosted Genomic Intelligence API
- **`run_gi_expression()`** — Predict gene expression from a TSS-centred window via the hosted Genomic Intelligence API
- **`run_gi_find_genes_and_predict_expression()`** — Annotate genes in a locus and predict expression for each, via the hosted Genomic Intelligence API
- **`run_gi_promoter()`** — Predict promoter regions in DNA via the hosted Genomic Intelligence API
- **`run_gi_splice()`** — Predict splice donor and acceptor sites via the hosted Genomic Intelligence API

## Setup

Every tool takes `sequences` and a config. Sequence length bounds are per task
and are checked locally against the endpoint's published floor before any
request is sent.

In [3]:
import os

from proto_tools.tools.sequence_scoring.genomic_intelligence import (
    GIAnnotationConfig,
    GIAnnotationInput,
    GIExpressionConfig,
    GIExpressionInput,
    GIFindGenesConfig,
    GIFindGenesInput,
    GIPromoterConfig,
    GIPromoterInput,
    GISpliceConfig,
    GISpliceInput,
    run_gi_annotation,
    run_gi_expression,
    run_gi_find_genes_and_predict_expression,
    run_gi_promoter,
    run_gi_splice,
)

assert os.environ.get("GI_API_KEY"), "set GI_API_KEY before running this notebook"

with open("hbb_locus.txt") as handle:
    HBB_LOCUS = handle.read().strip()

print(f"HBB locus: {len(HBB_LOCUS):,} bp (GRCh38 chr11:5,220,000-5,245,000, gene-sense)")

HBB locus: 25,001 bp (GRCh38 chr11:5,220,000-5,245,000, gene-sense)


## Promoter

Slides the model across the sequence and reports windows called as promoters.
Coordinates are 0-based with exclusive ends.

In [4]:
display_api_reference("gi-promoter", "input", "run_gi_promoter")

**Input** — `GIPromoterInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[GISequence]</code> | required | DNA sequences to score for promoter activity (>=300 bp each) |

In [5]:
display_api_reference("gi-promoter", "config", "run_gi_promoter")

**Config** — `GIPromoterConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gi_api_key</code> | <code>str &#124; None</code> | <code>$GI_API_KEY</code> | Bearer key for api.genomicintelligence.ai. Defaults to the GI_API_KEY env var if not set. |
| <code>model</code> | <code>str &#124; None</code> | <code>None</code> | Model id; leave unset to use the task's server-side default (GET /v1/tasks/{task}/models) |
| <code>respond_async</code> | <code>bool</code> | <code>False</code> | Send Prefer: respond-async and poll the job instead of waiting for a synchronous 200 |
| <code>timeout_seconds</code> | <code>float</code> | <code>1800.0</code> | Maximum wall-clock time to wait for an async job |
| <code>threshold</code> | <code>float</code> | <code>0.5</code> | Probability above which a window is reported as a promoter |

In [6]:
promoter = run_gi_promoter(
    GIPromoterInput(sequences=HBB_LOCUS[:5000]),
    GIPromoterConfig(),
)
result = promoter.results[0]
print(f"model            {result.meta.model}")
print(f"windows scored   {result.total_windows}")
print(f"called promoter  {result.promoter_windows}")
print(f"max probability  {result.max_probability:.4f}")

model            g0-promoter-2000bp
windows scored   5
called promoter  1
max probability  0.9504


In [7]:
display_api_reference("gi-promoter", "output", "run_gi_promoter")

**Output** — `GIPromoterOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[GIPromoterResult]</code> | required | One result per submitted sequence, in order |

**`GIPromoterResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>name</code> | <code>str</code> | required | Label supplied with the sequence |
| <code>sequence_length</code> | <code>int</code> | required | Length of the submitted sequence in base pairs |
| <code>promoter_windows</code> | <code>int</code> | required | Number of windows that cleared the threshold |
| <code>total_windows</code> | <code>int</code> | required | Number of windows scored |
| <code>max_probability</code> | <code>float &#124; None</code> | <code>None</code> | Highest window probability |
| <code>regions</code> | <code>list[PromoterRegion]</code> | <code>[]</code> | Regions called as promoters |
| <code>windows</code> | <code>list[PromoterWindow]</code> | <code>[]</code> | Every scored window |
| <code>meta</code> | <code>GIRequestMeta</code> | required | Provenance for the call |

**`PromoterRegion`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>start</code> | <code>int</code> | required | 0-based inclusive start in the submitted sequence |
| <code>end</code> | <code>int</code> | required | 0-based exclusive end in the submitted sequence |
| <code>score</code> | <code>float</code> | required | Model probability for the region |
| <code>name</code> | <code>str</code> | <code>''</code> | Service-assigned region label |

**`PromoterWindow`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>window_index</code> | <code>int</code> | required | Position of this window in the slide |
| <code>start</code> | <code>int</code> | required | 0-based inclusive start of the scored span |
| <code>end</code> | <code>int</code> | required | 0-based exclusive end of the scored span |
| <code>probability</code> | <code>float</code> | required | Model probability for the window |
| <code>is_positive</code> | <code>bool</code> | required | Whether the probability cleared the threshold |

**`GIRequestMeta`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>model</code> | <code>str</code> | required | Model the service resolved and ran |
| <code>request_id</code> | <code>str &#124; None</code> | <code>None</code> | Correlation id for support |
| <code>job_id</code> | <code>str &#124; None</code> | <code>None</code> | Identifier of the computation |
| <code>inference_time_ms</code> | <code>float &#124; None</code> | <code>None</code> | Server-side inference time in milliseconds |
| <code>cold_start</code> | <code>bool &#124; None</code> | <code>None</code> | Whether the model was loaded for this call |

## Splice sites

The model is strand-specific. Submit transcript orientation: the opposite strand
returns sites at different positions, often at high confidence, so the result
cannot be checked for orientation after the fact.

Each site is printed as a span because that is what it is. `start`/`end`
bounds one variable-width tokenizer token, and the exon/intron junction sits
somewhere inside it rather than on either endpoint.

In [8]:
display_api_reference("gi-splice", "input", "run_gi_splice")

**Input** — `GISpliceInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[GISequence]</code> | required | DNA sequences in transcript orientation (>=100 bp each) |

In [9]:
display_api_reference("gi-splice", "config", "run_gi_splice")

**Config** — `GISpliceConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gi_api_key</code> | <code>str &#124; None</code> | <code>$GI_API_KEY</code> | Bearer key for api.genomicintelligence.ai. Defaults to the GI_API_KEY env var if not set. |
| <code>model</code> | <code>str &#124; None</code> | <code>None</code> | Model id; leave unset to use the task's server-side default (GET /v1/tasks/{task}/models) |
| <code>respond_async</code> | <code>bool</code> | <code>False</code> | Send Prefer: respond-async and poll the job instead of waiting for a synchronous 200 |
| <code>timeout_seconds</code> | <code>float</code> | <code>1800.0</code> | Maximum wall-clock time to wait for an async job |
| <code>threshold</code> | <code>float</code> | <code>0.5</code> | Score above which a position is reported as a splice site |
| <code>site_types</code> | <code>list[Literal['donor', 'acceptor']] &#124; None</code> | <code>None</code> | Subset of ['donor', 'acceptor'] to report; unset reports both |

In [10]:
splice = run_gi_splice(
    GISpliceInput(sequences=HBB_LOCUS[:5000]),
    GISpliceConfig(threshold=0.5),
)
result = splice.results[0]
print(f"sites  {result.total_sites}  ({result.donor_sites} donor, {result.acceptor_sites} acceptor)")
for site in result.sites[:5]:
    print(f"  {site.site_type:<9} {site.start:>6}-{site.end:<6} {site.score:.4f}")

sites  2  (1 donor, 1 acceptor)
  acceptor    1668-1674   0.9998
  donor       1889-1896   0.9999


In [11]:
display_api_reference("gi-splice", "output", "run_gi_splice")

**Output** — `GISpliceOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[GISpliceResult]</code> | required | One result per submitted sequence, in order |

**`GISpliceResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>name</code> | <code>str</code> | required | Label supplied with the sequence |
| <code>sequence_length</code> | <code>int</code> | required | Length of the submitted sequence in base pairs |
| <code>total_sites</code> | <code>int</code> | required | Number of sites reported |
| <code>donor_sites</code> | <code>int</code> | required | Number of donor sites reported |
| <code>acceptor_sites</code> | <code>int</code> | required | Number of acceptor sites reported |
| <code>sites</code> | <code>list[SpliceSite]</code> | <code>[]</code> | The reported sites |
| <code>meta</code> | <code>GIRequestMeta</code> | required | Provenance for the call |

**`SpliceSite`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>name</code> | <code>str</code> | <code>''</code> | Service-assigned site label |
| <code>start</code> | <code>int</code> | required | 0-based inclusive start of the site's token span in the submitted sequence |
| <code>end</code> | <code>int</code> | required | 0-based exclusive end of the site's token span in the submitted sequence |
| <code>site_type</code> | <code>str</code> | required | 'donor' or 'acceptor' |
| <code>score</code> | <code>float</code> | required | Model score for the site |

**`GIRequestMeta`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>model</code> | <code>str</code> | required | Model the service resolved and ran |
| <code>request_id</code> | <code>str &#124; None</code> | <code>None</code> | Correlation id for support |
| <code>job_id</code> | <code>str &#124; None</code> | <code>None</code> | Identifier of the computation |
| <code>inference_time_ms</code> | <code>float &#124; None</code> | <code>None</code> | Server-side inference time in milliseconds |
| <code>cold_start</code> | <code>bool &#124; None</code> | <code>None</code> | Whether the model was loaded for this call |

## Annotation

Finds transcripts de novo, with no reference. Detection is strand-insensitive:
genes on either strand are found from one submission, and the reported `strand`
is relative to the sequence as submitted.

In [12]:
display_api_reference("gi-annotation", "input", "run_gi_annotation")

**Input** — `GIAnnotationInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[GISequence]</code> | required | DNA sequences to annotate (>=1,000 bp each) |

In [13]:
display_api_reference("gi-annotation", "config", "run_gi_annotation")

**Config** — `GIAnnotationConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gi_api_key</code> | <code>str &#124; None</code> | <code>$GI_API_KEY</code> | Bearer key for api.genomicintelligence.ai. Defaults to the GI_API_KEY env var if not set. |
| <code>model</code> | <code>str &#124; None</code> | <code>None</code> | Model id; leave unset to use the task's server-side default (GET /v1/tasks/{task}/models) |
| <code>respond_async</code> | <code>bool</code> | <code>False</code> | Send Prefer: respond-async and poll the job instead of waiting for a synchronous 200 |
| <code>timeout_seconds</code> | <code>float</code> | <code>1800.0</code> | Maximum wall-clock time to wait for an async job |
| <code>batch_size</code> | <code>int &#124; None</code> | <code>None</code> | Server-side batching hint; unset uses the service default |
| <code>reverse_complement</code> | <code>bool &#124; None</code> | <code>None</code> | Also scan the reverse complement; unset uses the service default |

In [14]:
annotation = run_gi_annotation(
    GIAnnotationInput(sequences=HBB_LOCUS),
    GIAnnotationConfig(),
)
result = annotation.results[0]
print(f"transcripts  {result.total_transcripts}")
for transcript in result.transcripts[:5]:
    print(f"  {transcript.name:<14} {transcript.start:>6}-{transcript.end:<6} "
          f"strand {transcript.strand}  TSS {transcript.tss_position}")

transcripts  2
  transcript_1    10516-12165  strand +  TSS 10516
  transcript_2    17927-19534  strand +  TSS 17927


In [15]:
display_api_reference("gi-annotation", "output", "run_gi_annotation")

**Output** — `GIAnnotationOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[GIAnnotationResult]</code> | required | One result per submitted sequence, in order |

**`GIAnnotationResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>name</code> | <code>str</code> | required | Label supplied with the sequence |
| <code>sequence_length</code> | <code>int</code> | required | Length of the submitted sequence in base pairs |
| <code>total_transcripts</code> | <code>int</code> | required | Number of transcripts found |
| <code>forward_strand</code> | <code>int</code> | <code>0</code> | Transcripts on the submitted orientation |
| <code>reverse_strand</code> | <code>int</code> | <code>0</code> | Transcripts on the opposite orientation |
| <code>transcripts</code> | <code>list[Transcript]</code> | <code>[]</code> | The transcripts found |
| <code>meta</code> | <code>GIRequestMeta</code> | required | Provenance for the call |

**`Transcript`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>name</code> | <code>str</code> | <code>''</code> | Service-assigned transcript label |
| <code>start</code> | <code>int</code> | required | 0-based inclusive start in the submitted sequence |
| <code>end</code> | <code>int</code> | required | 0-based exclusive end in the submitted sequence |
| <code>strand</code> | <code>str</code> | <code>''</code> | Orientation relative to the submitted sequence |
| <code>score</code> | <code>float</code> | <code>0.0</code> | Model confidence for the transcript |
| <code>tss_position</code> | <code>int &#124; None</code> | <code>None</code> | Transcription start site offset |
| <code>polya_position</code> | <code>int &#124; None</code> | <code>None</code> | Poly(A) site offset |
| <code>transcript_type</code> | <code>str &#124; None</code> | <code>None</code> | Biotype, when the model reports one |

**`GIRequestMeta`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>model</code> | <code>str</code> | required | Model the service resolved and ran |
| <code>request_id</code> | <code>str &#124; None</code> | <code>None</code> | Correlation id for support |
| <code>job_id</code> | <code>str &#124; None</code> | <code>None</code> | Identifier of the computation |
| <code>inference_time_ms</code> | <code>float &#124; None</code> | <code>None</code> | Server-side inference time in milliseconds |
| <code>cold_start</code> | <code>bool &#124; None</code> | <code>None</code> | Whether the model was loaded for this call |

## Expression

The model scores exactly one 9,198 bp window centred on a TSS. Submit that
window, or a longer locus plus `tss_index` and let the service cut it — the
window actually scored is echoed back.

`description` is conditioning text fed to the model, not a label. Its wording
changes the prediction, so hold it fixed across runs you intend to compare.

In [16]:
display_api_reference("gi-expression", "input", "run_gi_expression")

**Input** — `GIExpressionInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[ExpressionSequence]</code> | required | Loci to score, each a 9,198 bp TSS window or a longer locus plus tss_index |

In [17]:
display_api_reference("gi-expression", "config", "run_gi_expression")

**Config** — `GIExpressionConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gi_api_key</code> | <code>str &#124; None</code> | <code>$GI_API_KEY</code> | Bearer key for api.genomicintelligence.ai. Defaults to the GI_API_KEY env var if not set. |
| <code>model</code> | <code>str &#124; None</code> | <code>None</code> | Model id; leave unset to use the task's server-side default (GET /v1/tasks/{task}/models) |
| <code>respond_async</code> | <code>bool</code> | <code>False</code> | Send Prefer: respond-async and poll the job instead of waiting for a synchronous 200 |
| <code>timeout_seconds</code> | <code>float</code> | <code>1800.0</code> | Maximum wall-clock time to wait for an async job |
| <code>description</code> | <code>str</code> | <code>'assay term name is polyA plus RNA-seq. biosample summary is Homo sapiens K562.'</code> | Experimental context fed to the model; wording changes the prediction |

In [18]:
tss = result.transcripts[0].tss_position if result.transcripts else len(HBB_LOCUS) // 2

expression = run_gi_expression(
    GIExpressionInput(sequences=[{"sequence": HBB_LOCUS, "name": "HBB", "tss_index": tss}]),
    GIExpressionConfig(),
)
prediction = expression.results[0]
print(f"tss_index applied  {prediction.tss_index}")
print(f"window scored      {prediction.scored_window}")
print(f"log(TPM+1)         {prediction.expression_log_tpm:.4f}")
print(f"TPM                {prediction.expression_tpm:.4f}")

tss_index applied  10516
window scored      [5917, 15115]
log(TPM+1)         1.0703
TPM                1.9163


In [19]:
display_api_reference("gi-expression", "output", "run_gi_expression")

**Output** — `GIExpressionOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[ExpressionPrediction]</code> | required | One result per submitted locus, in order |

**`ExpressionPrediction`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>name</code> | <code>str</code> | required | Label supplied with the sequence |
| <code>sequence_length</code> | <code>int</code> | required | Length of the submitted sequence in base pairs |
| <code>expression_log_tpm</code> | <code>float &#124; None</code> | <code>None</code> | Predicted log(TPM+1) |
| <code>expression_tpm</code> | <code>float &#124; None</code> | <code>None</code> | Predicted TPM |
| <code>tss_index</code> | <code>int &#124; None</code> | <code>None</code> | TSS offset the service applied |
| <code>scored_window</code> | <code>list[int] &#124; None</code> | <code>None</code> | Window actually scored, as [start, end) |
| <code>meta</code> | <code>GIRequestMeta</code> | required | Provenance for the call |

**`GIRequestMeta`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>model</code> | <code>str</code> | required | Model the service resolved and ran |
| <code>request_id</code> | <code>str &#124; None</code> | <code>None</code> | Correlation id for support |
| <code>job_id</code> | <code>str &#124; None</code> | <code>None</code> | Identifier of the computation |
| <code>inference_time_ms</code> | <code>float &#124; None</code> | <code>None</code> | Server-side inference time in milliseconds |
| <code>cold_start</code> | <code>bool &#124; None</code> | <code>None</code> | Whether the model was loaded for this call |

### Conditioning text changes the prediction

The same sequence under two descriptions. This is why the wording has to be held
fixed when comparing runs.

In [20]:
for description in (
    "assay term name is polyA plus RNA-seq. biosample summary is Homo sapiens K562.",
    "K562",
):
    out = run_gi_expression(
        GIExpressionInput(sequences=[{"sequence": HBB_LOCUS, "name": "HBB", "tss_index": tss}]),
        GIExpressionConfig(description=description),
    )
    print(f"{out.results[0].expression_log_tpm:8.4f}  <- {description[:60]}")

  1.0703  <- assay term name is polyA plus RNA-seq. biosample summary is 


  0.1426  <- K562


## Find genes and predict expression

Annotation and expression in one call, centring each window on the gene's own
TSS. Use it when the TSS positions are not known in advance.

This endpoint refuses synchronous delivery above 50,000 bp; the tool switches to
the polling path automatically. Its length ceiling is the endpoint's own
500,000 bp, which is not the expression model's window.

In [21]:
display_api_reference("gi-find-genes-and-predict-expression", "input", "run_gi_find_genes_and_predict_expression")

**Input** — `GIFindGenesInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[GISequence]</code> | required | Genomic loci to annotate and score (>=1,000 bp each) |

In [22]:
display_api_reference("gi-find-genes-and-predict-expression", "config", "run_gi_find_genes_and_predict_expression")

**Config** — `GIFindGenesConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gi_api_key</code> | <code>str &#124; None</code> | <code>$GI_API_KEY</code> | Bearer key for api.genomicintelligence.ai. Defaults to the GI_API_KEY env var if not set. |
| <code>model</code> | <code>str &#124; None</code> | <code>None</code> | Model id; leave unset to use the task's server-side default (GET /v1/tasks/{task}/models) |
| <code>respond_async</code> | <code>bool</code> | <code>False</code> | Send Prefer: respond-async and poll the job instead of waiting for a synchronous 200 |
| <code>timeout_seconds</code> | <code>float</code> | <code>1800.0</code> | Maximum wall-clock time to wait for an async job |
| <code>description</code> | <code>str</code> | <code>'assay term name is polyA plus RNA-seq. biosample summary is Homo sapiens K562.'</code> | Experimental context applied to every gene found; wording changes the predictions |
| <code>annotation_model</code> | <code>str &#124; None</code> | <code>None</code> | Annotation-stage model id; unset uses the service default |
| <code>expression_model</code> | <code>str &#124; None</code> | <code>None</code> | Expression-stage model id; unset uses the service default |

In [23]:
workflow = run_gi_find_genes_and_predict_expression(
    GIFindGenesInput(sequences=[{"sequence": HBB_LOCUS, "name": "HBB"}]),
    GIFindGenesConfig(),
)
result = workflow.results[0]
print(f"genes found   {result.genes_found}")
print(f"genes scored  {result.genes_scored}")
for gene in result.predictions:
    if gene.skipped:
        print(f"  {gene.gene_name:<14} skipped: {gene.skip_reason}")
    else:
        print(f"  {gene.gene_name:<14} TSS {gene.tss_position:>6}  log(TPM+1) {gene.expression:.4f}")

genes found   2
genes scored  2
  transcript_1   TSS  10516  log(TPM+1) 1.0703
  transcript_2   TSS  17927  log(TPM+1) 0.9570


In [24]:
display_api_reference("gi-find-genes-and-predict-expression", "output", "run_gi_find_genes_and_predict_expression")

**Output** — `GIFindGenesOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[GIFindGenesResult]</code> | required | One result per submitted locus, in order |

**`GIFindGenesResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>name</code> | <code>str</code> | required | Label supplied with the sequence |
| <code>sequence_length</code> | <code>int</code> | required | Length of the submitted sequence in base pairs |
| <code>genes_found</code> | <code>int</code> | required | Genes the annotation stage reported |
| <code>genes_scored</code> | <code>int</code> | required | Genes expression was predicted for |
| <code>predictions</code> | <code>list[GenePrediction]</code> | <code>[]</code> | One record per gene found |
| <code>meta</code> | <code>GIRequestMeta</code> | required | Provenance for the call |

**`GenePrediction`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gene_index</code> | <code>int</code> | required | Position of the gene in the annotation output |
| <code>gene_name</code> | <code>str</code> | <code>''</code> | Service-assigned gene label |
| <code>strand</code> | <code>str</code> | <code>''</code> | Orientation relative to the submitted sequence |
| <code>tss_position</code> | <code>int</code> | <code>0</code> | TSS offset in the submitted sequence |
| <code>expression</code> | <code>float &#124; None</code> | <code>None</code> | Predicted log(TPM+1) |
| <code>expression_tpm</code> | <code>float &#124; None</code> | <code>None</code> | Predicted TPM |
| <code>skipped</code> | <code>bool</code> | <code>False</code> | Whether expression was skipped for this gene |
| <code>skip_reason</code> | <code>str &#124; None</code> | <code>None</code> | Why expression was skipped |

**`GIRequestMeta`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>model</code> | <code>str</code> | required | Model the service resolved and ran |
| <code>request_id</code> | <code>str &#124; None</code> | <code>None</code> | Correlation id for support |
| <code>job_id</code> | <code>str &#124; None</code> | <code>None</code> | Identifier of the computation |
| <code>inference_time_ms</code> | <code>float &#124; None</code> | <code>None</code> | Server-side inference time in milliseconds |
| <code>cold_start</code> | <code>bool &#124; None</code> | <code>None</code> | Whether the model was loaded for this call |

## Scoring many sequences

Every tool takes a list and returns one result per input, in order, which is the
shape a Constraint or Optimizer consumes when scoring a population.

In [25]:
variants = [
    {"sequence": HBB_LOCUS[offset : offset + 3000], "name": f"window_{offset}"}
    for offset in (0, 6000, 12000)
]
batch = run_gi_promoter(GIPromoterInput(sequences=variants), GIPromoterConfig())
for item in batch.results:
    print(f"{item.name:<14} max probability {item.max_probability:.4f}")

window_0       max probability 0.9504
window_6000    max probability 0.0117
window_12000   max probability 0.2551
